In [ ]:
!pip install -q sentence-transformers numpy

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
with open(file_name, "r", encoding="utf-8") as f:
    text = f.read()

print("File loaded:", file_name)
print("Total characters:", len(text))
print("Number of paragraphs:", len([p for p in text.split("\n") if p.strip()]))

Saving World_cities.txt to World_cities.txt
File loaded: World_cities.txt
Total characters: 4444
Number of paragraphs: 13


In [ ]:
CHUNK_SIZE = 500
OVERLAP_PERCENT = 0.15
OVERLAP = int(CHUNK_SIZE * OVERLAP_PERCENT)
STEP = CHUNK_SIZE - OVERLAP

def chunk_text(text, chunk_size, step):
    chunks = []
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks

chunks = chunk_text(text, CHUNK_SIZE, STEP)
print("Chunk size:", CHUNK_SIZE, "| Overlap:", OVERLAP, "characters")
print("Number of chunks:", len(chunks))

Chunk size: 500 | Overlap: 75 characters
Number of chunks: 11


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(chunks)
print("Embeddings shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (11, 384)


In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, top_k=3):

    query_embedding = model.encode(query)

    scores = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]

    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

    print("Question:", query)
    print("=" * 80)
    for rank, (idx, score) in enumerate(ranked, start=1):
        print(f"\nAnswer {rank} (Chunk #{idx}) - Similarity Score: {score:.4f}")
        print("-" * 80)
        print(chunks[idx])
    print("\n" + "=" * 80)


query = input("Enter your question: ")
search(query)

Enter your question: what is the largest city based on population ?
Question: what is the largest city based on population ?

Answer 1 (Chunk #9) - Similarity Score: 0.6749
--------------------------------------------------------------------------------
and Karachi, each with populations above 20 million in broader metropolitan estimates. The world's urban population continues to grow quickly, and experts expect many of the biggest future cities to appear in Asia and Africa.

Ranking of the most populous cities in the world by population: the number one largest city in the world by population is Jakarta, Indonesia, with about 42 million people. The second most populous city is Dhaka, Bangladesh, with about 37 million people. The third most pop

Answer 2 (Chunk #10) - Similarity Score: 0.6196
--------------------------------------------------------------------------------
city is Dhaka, Bangladesh, with about 37 million people. The third most populous city is Tokyo, Japan, with about 33